In [1]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

csv_file = "./filtered_urls/categories_urls.csv"
output_folder = "./database/categories_json_db"
os.makedirs(output_folder, exist_ok=True)

# Selenium options with better stealth
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def scrape_newfind_page(url, soup):
    """Scrape brand/model type parts listing page (Newfind page)"""
    newfind_data = {}
    newfind_data['url'] = url
    newfind_data['page_type'] = 'newfind'

    # Page metadata
    page_container = soup.find('div', attrs={'data-page-type': 'Newfind'})
    if page_container:
        newfind_data['brand'] = page_container.get('data-brand')
        newfind_data['model_type'] = page_container.get('data-modeltype')
    else:
        newfind_data['brand'] = None
        newfind_data['model_type'] = None

    # Page title
    title = soup.find('h1', class_='title-main')
    newfind_data['page_title'] = title.text.strip() if title else None

    # Brand image
    brand_img = soup.find('div', class_='nf__brand')
    if brand_img:
        img_tag = brand_img.find('img')
        newfind_data['brand_image'] = img_tag.get('src') if img_tag else None
    else:
        newfind_data['brand_image'] = None

    # Popular parts section
    popular_parts = []
    part_elements = soup.find_all('div', class_='nf__part')
    
    for part_elem in part_elements:
        part_data = {}
        
        # Part title/name
        title_elem = part_elem.find('a', class_='nf__part__detail__title')
        if title_elem:
            part_data['name'] = title_elem.find('span').text.strip() if title_elem.find('span') else None
            part_data['url'] = 'https://www.partselect.com' + title_elem.get('href') if title_elem.get('href') else None
        else:
            part_data['name'] = None
            part_data['url'] = None

        # Part numbers
        part_numbers = part_elem.find_all('div', class_='nf__part__detail__part-number')
        if len(part_numbers) >= 1:
            ps_num = part_numbers[0].find('strong')
            part_data['partselect_number'] = ps_num.text.strip() if ps_num else None
        else:
            part_data['partselect_number'] = None
            
        if len(part_numbers) >= 2:
            mfr_num = part_numbers[1].find('strong')
            part_data['manufacturer_part_number'] = mfr_num.text.strip() if mfr_num else None
        else:
            part_data['manufacturer_part_number'] = None

        # Price information
        price_elem = part_elem.find('div', class_='price')
        if price_elem:
            currency = price_elem.find('span', class_='price__currency')
            price_text = price_elem.get_text(strip=True).replace('$', '').strip()
            part_data['price'] = price_text if price_text else None
        else:
            part_data['price'] = None

        # Original price (if on sale)
        original_price = part_elem.find('div', class_='original-price')
        part_data['original_price'] = original_price.get_text(strip=True).replace('$', '').strip() if original_price else None

        # Discount badge
        discount_badge = part_elem.find('div', class_='price__discount-badge')
        if discount_badge:
            discount_text = discount_badge.find('span')
            part_data['discount'] = discount_text.text.strip() if discount_text else None
        else:
            part_data['discount'] = None

        # Stock status
        stock_elem = part_elem.find('div', class_='nf__part__left-col__basic-info__stock')
        if stock_elem:
            stock_span = stock_elem.find('span')
            part_data['stock_status'] = stock_span.text.strip() if stock_span else None
        else:
            part_data['stock_status'] = None

        # Rating and reviews
        rating_elem = part_elem.find('div', class_='rating')
        if rating_elem:
            # Extract rating from star width percentage
            stars_upper = rating_elem.find('div', class_='rating__stars__upper')
            if stars_upper:
                style = stars_upper.get('style', '')
                if 'width:' in style:
                    percentage = style.split('width:')[1].split('%')[0].strip()
                    try:
                        part_data['rating'] = round(float(percentage) / 20, 2)  # Convert to 5-star scale
                    except:
                        part_data['rating'] = None
                else:
                    part_data['rating'] = None
            else:
                part_data['rating'] = None
            
            # Review count
            review_count = rating_elem.find('span', class_='rating__count')
            part_data['review_count'] = review_count.text.strip() if review_count else None
        else:
            part_data['rating'] = None
            part_data['review_count'] = None

        # Part image
        img_elem = part_elem.find('img', class_='b-lazy')
        part_data['image_url'] = img_elem.get('data-src') if img_elem and img_elem.get('data-src') else None

        # Video indicator
        video_sticker = part_elem.find('div', class_='nf__part__left-col__img__stickers__video')
        part_data['has_video'] = True if video_sticker else False

        # Description
        desc_div = part_elem.find('div', class_='nf__part__detail')
        if desc_div:
            # Get text after the part numbers but before symptoms section
            desc_text = []
            for content in desc_div.children:
                if hasattr(content, 'get') and content.get('class'):
                    if 'nf__part__detail__symptoms' in content.get('class', []):
                        break
                    if 'nf__part__detail__instruction' in content.get('class', []):
                        break
                elif hasattr(content, 'string') and content.string:
                    text = content.string.strip()
                    if text:
                        desc_text.append(text)
            part_data['description'] = ' '.join(desc_text) if desc_text else None
        else:
            part_data['description'] = None

        # Symptoms fixed
        symptoms = []
        symptoms_section = part_elem.find('div', class_='nf__part__detail__symptoms')
        if symptoms_section:
            symptom_list = symptoms_section.find('ul')
            if symptom_list:
                for li in symptom_list.find_all('li'):
                    # Skip the "See more..." link
                    if not li.find('a'):
                        symptoms.append(li.text.strip())
        part_data['fixes_symptoms'] = symptoms

        # Installation instructions/repair story
        instruction_section = part_elem.find('div', class_='nf__part__detail__instruction')
        if instruction_section:
            author = instruction_section.find('div', class_='nf__part__detail__instruction__creator')
            quote_div = instruction_section.find('div', class_='nf__part__detail__instruction__quote')
            
            if quote_div:
                quote_title = quote_div.find('div', class_='bold')
                quote_text = quote_div.find('span', class_='d-block')
                
                part_data['installation_story'] = {
                    'author': author.text.strip() if author else None,
                    'title': quote_title.text.strip() if quote_title else None,
                    'description': quote_text.text.strip() if quote_text else None
                }
            else:
                part_data['installation_story'] = None
        else:
            part_data['installation_story'] = None

        popular_parts.append(part_data)
    
    newfind_data['popular_parts'] = popular_parts

    # Appliance types section
    appliance_types = []
    brand_section = soup.find('h2', id='ShopByBrand')
    if brand_section:
        links_ul = brand_section.find_next('ul', class_='nf__links')
        if links_ul:
            for li in links_ul.find_all('li'):
                link = li.find('a')
                if link:
                    appliance_types.append({
                        'name': link.text.strip(),
                        'url': 'https://www.partselect.com' + link.get('href') if link.get('href') else None
                    })
    newfind_data['appliance_types'] = appliance_types

    # Related parts categories
    related_parts_categories = []
    part_type_section = soup.find('h2', id='ShopByPartType')
    if part_type_section:
        links_ul = part_type_section.find_next('ul', class_='nf__links')
        if links_ul:
            for li in links_ul.find_all('li'):
                link = li.find('a')
                if link:
                    related_parts_categories.append({
                        'name': link.text.strip(),
                        'url': 'https://www.partselect.com' + link.get('href') if link.get('href') else None
                    })
    newfind_data['related_parts_categories'] = related_parts_categories

    # Popular models
    popular_models = []
    models_section = soup.find('h2', id='TopModelsSectionTitle')
    if models_section:
        # Get the next ul.nf__links after the search box
        current = models_section.find_next('ul', class_='nf__links')
        if current:
            for li in current.find_all('li'):
                link = li.find('a')
                if link:
                    popular_models.append({
                        'model_number': link.text.strip(),
                        'url': 'https://www.partselect.com' + link.get('href') if link.get('href') else None
                    })
    newfind_data['popular_models'] = popular_models

    return newfind_data

# Load URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        urls = [row['url'] for row in reader]
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    driver.quit()
    exit(1)

print(f"Found {len(urls)} URLs to scrape")

for idx, url in enumerate(urls, 1):
    try:
        print(f"\n[{idx}/{len(urls)}] Processing: {url}")
        
        driver.get(url)
        
        # Wait for page to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")
        
        # Random delay
        time.sleep(random.uniform(2, 4))
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Scrape the newfind page
        data = scrape_newfind_page(url, soup)

        # Generate filename from brand and model type
        parts = []
        if data.get('brand'):
            parts.append(data['brand'])
        if data.get('model_type'):
            parts.append(data['model_type'])
        
        filename = '-'.join(parts) if parts else url.split('/')[-1].replace('.htm', '')
        
        # Clean filename
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        filename = filename + ".json"
        filepath = os.path.join(output_folder, filename)

        # Save to JSON
        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {filename} (newfind)")
        print(f"    - Popular parts: {len(data.get('popular_parts', []))}")
        print(f"    - Appliance types: {len(data.get('appliance_types', []))}")
        print(f"    - Related categories: {len(data.get('related_parts_categories', []))}")
        print(f"    - Popular models: {len(data.get('popular_models', []))}")

    except Exception as e:
        print(f"  ✗ Failed for {url}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue
    
    # Random delay between requests
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✓ All done! Scraped data saved to:", output_folder)

Found 74 URLs to scrape

[1/74] Processing: https://www.partselect.com/White-Westinghouse-Refrigerator-Parts.htm
  ✓ Saved Westinghouse-Refrigerator.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 28
    - Popular models: 20

[2/74] Processing: https://www.partselect.com/White-Westinghouse-Dishwasher-Parts.htm
  ✓ Saved Westinghouse-Dishwasher.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 2
    - Popular models: 20

[3/74] Processing: https://www.partselect.com/Whirlpool-Refrigerator-Parts.htm
  ✓ Saved Whirlpool-Refrigerator.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 44
    - Popular models: 20

[4/74] Processing: https://www.partselect.com/Whirlpool-Dishwasher-Parts.htm
  ✓ Saved Whirlpool-Dishwasher.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 34
    - Popular models: 20

[5/74] Processing: https://www.parts

  ✓ Saved Kelvinator-Dishwasher.json (newfind)
    - Popular parts: 10
    - Appliance types: 8
    - Related categories: 10
    - Popular models: 20

[38/74] Processing: https://www.partselect.com/Jenn-Air-Refrigerator-Parts.htm
  ✓ Saved Jenn-Air-Refrigerator.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 35
    - Popular models: 20

[39/74] Processing: https://www.partselect.com/Jenn-Air-Dishwasher-Parts.htm
  ✓ Saved Jenn-Air-Dishwasher.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 22
    - Popular models: 20

[40/74] Processing: https://www.partselect.com/International-Refrigerator-Parts.htm
  ✓ Saved International-Refrigerator.json (newfind)
    - Popular parts: 10
    - Appliance types: 0
    - Related categories: 7
    - Popular models: 20

[41/74] Processing: https://www.partselect.com/Inglis-Refrigerator-Parts.htm
  ✓ Saved Inglis-Refrigerator.json (newfind)
    - Popular parts: 10
    - Ap


[74/74] Processing: https://www.partselect.com/Admiral-Dishwasher-Parts.htm
  ✓ Saved Admiral-Dishwasher.json (newfind)
    - Popular parts: 10
    - Appliance types: 2
    - Related categories: 10
    - Popular models: 20

✓ All done! Scraped data saved to: ./database/categories_json_db
